# W7 Lab B — Bayesian Functions and Predictive Uncertainty

**October 13, 2026 · CPU · no manual upload, API key or GPU.** Save a copy in Drive and run from the top. This notebook runs independently of the other lab. If setup asks for a runtime restart after installing packages, restart once and run all again.

[Main note](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/course/notion/w7/w7-predictive-models.md) · [Worked calculations](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/course/notion/w7/w7-worked-calculations.md) · [Worksheet](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/course/notion/w7/w7-experiment-record.md) · [API guide](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/course/notion/w7/w7-code-guide.md)

Predict before running. Unfinished exercise cells do not block the demonstrations. Fixed code references and hashes make the imported implementation inspectable. Code links name the exact definitions used here.


In [ ]:
import sys, subprocess, importlib.metadata, hashlib, urllib.request, time
from pathlib import Path

PINS = {"numpy":"2.2.6", "matplotlib":"3.10.3", "scipy":"1.16.3",
        "pandas":"2.3.3", "gradio":"6.27.0"}
missing = []
for package, version in PINS.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        missing.append(f"{package}=={version}")
restart = [p for p,v in PINS.items() if p in sys.modules and getattr(sys.modules[p], "__version__", v) != v]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
if restart:
    raise RuntimeError("Packages updated. Use Runtime > Restart session, then Run all. Loaded old packages: "+", ".join(restart))

CODE_REF = "2026-fall-w7"
EXPECTED = {'advml_predictive.py': 'beaa85083c3f1cc4ead5f253ff11c82b7f71dc3ad2da8fc21d7307505e20583b', 'advml_predictive_app.py': 'aa0fd860bdddfef37fab03578b342de381744237771d81b72fd65288fa93cec9'}
local = Path.cwd() / "src"
local_ok = all((local/name).is_file() and hashlib.sha256((local/name).read_bytes()).hexdigest()==digest
               for name,digest in EXPECTED.items())
support = local if local_ok else Path.cwd() / "w7_support" / CODE_REF
support.mkdir(parents=True, exist_ok=True)
downloaded = []
for name,digest in EXPECTED.items():
    path = support/name
    if path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest()==digest:
        continue
    url = f"https://raw.githubusercontent.com/lunalab-ai/advML/{CODE_REF}/src/{name}"
    for attempt in range(3):
        try:
            with urllib.request.urlopen(url, timeout=40) as response:
                payload = response.read()
            break
        except (OSError, TimeoutError):
            if attempt == 2:
                raise
            time.sleep(attempt+1)
    if hashlib.sha256(payload).hexdigest()!=digest:
        raise RuntimeError(f"Source hash mismatch: {name}. Do not run unverified code.")
    temporary = path.with_suffix(".download")
    temporary.write_bytes(payload)
    temporary.replace(path)
    downloaded.append(name)
sys.path.insert(0, str(support.resolve()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
from IPython.display import display
print({"Python":sys.version.split()[0], "code_ref":CODE_REF, "downloaded":downloaded,
       "cache_or_reviewed_local":not downloaded, "versions":PINS})


## 1. One unknown slope, two kinds of prediction

[BayesianLinear: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/src/advml_predictive.py#L84) · [BayesianLinear.fit: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/src/advml_predictive.py#L89) · [BayesianLinear.predict: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/src/advml_predictive.py#L100) · [total_variance: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/src/advml_predictive.py#L67)

Start with prior variance 1 and known noise variance 1. The features contain only x, so there is no intercept in this tiny example. After fitting, predict at x=3. Read the mean, latent variance and future-response variance separately.

$$\operatorname{Var}(y_*\mid D)=\operatorname{Var}(f_*\mid D)+\sigma^2.$$


In [ ]:
from advml_predictive import (BayesianLinear,GaussianProcess,rbf_kernel,
    total_variance,make_data,draw_functions,evaluate_predictions)
from advml_predictive_app import run_experiment,fit_prediction,create_app,MODELS
slope=BayesianLinear(noise=1).fit(np.array([[1.],[2.]]),np.array([1.,2.]))
tiny=slope.predict(np.array([[3.]]))
print({'posterior_weight_mean':slope.mean,'posterior_weight_covariance':slope.cov,
       'new_function_mean':tiny['mean'],'function_variance':tiny['var_f'],
       'new_response_variance':tiny['var_y']})


### B1 — Your turn

Using the same slope posterior, predict at x=2. Calculate the latent mean, latent variance and future-response variance. A classmate adds the noise SD to the latent SD to obtain a total SD. Explain and fix the mistake.


In [ ]:
b1_response = ""  # Response field for B1; later demonstrations remain runnable.


## 2. Within-model and between-model variance

The finite mixture below is a calculation you can check by hand. Its weights sum to one. They are not a training dataset. The two variance components answer different questions.


In [ ]:
mixture=total_variance([.2,.3,.5],[0,1,2],[.25,.25,.25])
display(pd.DataFrame([mixture]))
fig,ax=plt.subplots(figsize=(6,3))
ax.bar(['Within','Between','Total'],[mixture['within'],mixture['between'],mixture['total']])
ax.set(ylabel='Variance');plt.show()


### B2 — Your turn

For equal mixture weights [.5,.5], component means [-1,1] and component variances [.5,.5], calculate the predictive mean and both variance components. What changes if you estimate these fixed quantities with more Monte Carlo draws?


In [ ]:
b2_response = ""  # Response field for B2; later demonstrations remain runnable.


## 3. A kernel creates a joint distribution

[rbf_kernel: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/src/advml_predictive.py#L108) · [GaussianProcess: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/src/advml_predictive.py#L116) · [GaussianProcess.fit: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/src/advml_predictive.py#L122) · [GaussianProcess.predict: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/src/advml_predictive.py#L131) · [draw_functions: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/src/advml_predictive.py#L203)

The RBF length scale controls covariance across input distances. The amplitude is a function SD. Draw complete curves using the joint covariance, not independent draws from each point’s marginal. Then condition the GP on two noisy observations.


In [ ]:
grid=np.linspace(-2,3,121)
K=rbf_kernel(grid,grid,length=1)
prior_curves=draw_functions(np.zeros(len(grid)),K,draws=4,seed=5)
gp_tiny=GaussianProcess(noise=.5).fit([0,1],[0,1])
post=gp_tiny.predict(grid)
posterior_curves=draw_functions(post['mean'],post['cov_f'],draws=4,seed=5)
fig,axes=plt.subplots(1,2,figsize=(9,3.5))
axes[0].plot(grid,prior_curves.T);axes[0].set(title='Prior functions',xlabel='x',ylabel='f')
axes[1].plot(grid,posterior_curves.T);axes[1].scatter([0,1],[0,1],c='black')
axes[1].set(title='Posterior functions: noisy observations',xlabel='x',ylabel='f')
plt.tight_layout();plt.show()
print('Two-point worked-example prediction:',gp_tiny.predict([.5]))


### B3 — Your turn

Why need the posterior curves not pass exactly through the two observed responses? Explain why sampling each grid point independently would be wrong. Is a kernel invalid simply because one covariance entry is negative?


In [ ]:
b3_response = ""  # Response field for B3; later demonstrations remain runnable.


## 4. Compute the same model in weight space and function space

We include an intercept and a slope, with prior covariance 0.7 times the identity. The induced kernel is `0.7 * Phi @ Phi.T`. Every modeling choice must match. This tiny calculation uses an independent linear solve as a cross-check of the class implementation.


In [ ]:
x=np.array([-1.,0.,2.]);y=np.array([1.,-.2,1.2]);xt=np.linspace(-2,3,41)
Phi=np.column_stack([np.ones(len(x)),x]);Psi=np.column_stack([np.ones(len(xt)),xt])
prior_variance=.7;noise=.3
weight_model=BayesianLinear(noise,prior_variance).fit(Phi,y)
weight_prediction=weight_model.predict(Psi)
K_train=prior_variance*Phi@Phi.T
cross=prior_variance*Psi@Phi.T
C=K_train+noise**2*np.eye(len(x))
function_mean=cross@np.linalg.solve(C,y)
function_cov=prior_variance*Psi@Psi.T-cross@np.linalg.solve(C,cross.T)
print({'maximum_mean_difference':np.max(abs(function_mean-weight_prediction['mean'])),
       'maximum_covariance_difference':np.max(abs(function_cov-weight_prediction['cov_f'])),
       'negative_but_valid_covariance':K_train[0,2]})


### B4 — Your turn

Change the weight-space prior variance to 1.4 but leave the kernel calculation unchanged. Should the predictions still agree? Record their discrepancy, identify the modeling mismatch, and restore a consistent comparison.


In [ ]:
b4_response = ""  # Response field for B4; later demonstrations remain runnable.


## 5. Compare models on the same held-out outcomes

[make_data: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/src/advml_predictive.py#L26) · [fit_prediction: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/src/advml_predictive_app.py#L14) · [evaluate_predictions: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/src/advml_predictive.py#L214)

All methods receive the same training pairs. The test grid and independently noisy test responses are held out. The GP uses fixed hyperparameters; the network optimizer uses only training data. Neural-linear means exact last-layer inference conditional on learned features, not full-network Bayesian inference. The MLP’s response band includes known noise only.


In [ ]:
data=make_data(32,.2,19)
rows=[]
for model in MODELS:
    prediction,status=fit_prediction(model,data)
    rows.append(dict(model=model,**evaluate_predictions(data['yt'],prediction['mean'],prediction['var_y'])))
    print(model,':',status)
comparison=pd.DataFrame(rows);display(comparison)
figure,regions,assumptions,csv_path=run_experiment('GP (RBF)')
display(figure);display(regions);print(assumptions)


### B5 — Your turn

Use the region rows to compare Bayesian linear regression with the GP. Explain why a narrow interval can have poor coverage. Is the apparent best model in this one simulator guaranteed to be best for new applications?


In [ ]:
b5_response = ""  # Response field for B5; later demonstrations remain runnable.


## 6. Add observations while holding GP assumptions fixed

For a fixed seed and noise SD the larger training set contains the smaller one. Conditional Gaussian variance should not increase solely because we add observations at new inputs. This statement does not promise that every individual prediction error falls.


In [ ]:
small=make_data(16,.2,19);large=make_data(48,.2,19)
ps=GaussianProcess().fit(small['x'],small['y']).predict(small['xt'])
pl=GaussianProcess().fit(large['x'],large['y']).predict(large['xt'])
print({'largest_latent_variance_increase':np.max(pl['var_f']-ps['var_f']),
       'mean_variance_small':ps['var_f'].mean(),'mean_variance_large':pl['var_f'].mean()})
fig,ax=plt.subplots(figsize=(8,3))
ax.plot(small['xt'],ps['var_f'],label='16 observations')
ax.plot(large['xt'],pl['var_f'],label='48 observations')
ax.set(xlabel='x',ylabel='Latent variance');ax.legend();plt.show()


### B6 — Your turn

Compare length scales .3, 1.0 and 2.5 on the same 32 observations. Predict the effect first. Report the mean width and coverage in the gap and outside the training range. Explain why choosing the best setting from this table and reporting the same table as a final test would be misleading.


In [ ]:
b6_response = ""  # Response field for B6; later demonstrations remain runnable.


## 7. Extend the web workbench

[run_experiment: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/src/advml_predictive_app.py#L38) · [create_app: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/src/advml_predictive_app.py#L88) · [likelihood_demo: definition](https://github.com/lunalab-ai/advML/blob/2026-fall-w7/src/advml_predictive_app.py#L72)

The callback receives one region’s metrics and can add a field with a new name. `run_experiment` returns figure, metric table, status and CSV path. `create_app` builds the UI; launch is a separate step. Your extension should express a statistical idea, not require knowledge of the UI library.


In [ ]:
def student_extra(metrics):
    return {}  # Add a new diagnostic without replacing existing fields.

figure,table,status,exported=run_experiment(extra=student_extra)
display(table);print('CSV prepared:',exported)


### B7 — Your turn

Add `miss_rate`, the fraction of outcomes outside the reported 95% interval, to the callback. Explain its relation to coverage. Run the app, change a setting, export the CSV and verify that this new column is present. Does a miss rate close to .05 on one finite test grid prove calibration?


In [ ]:
b7_response = ""  # Response field for B7; later demonstrations remain runnable.


In [ ]:
np.testing.assert_allclose(weight_prediction['mean'],function_mean,atol=1e-10)
np.testing.assert_allclose(weight_prediction['cov_f'],function_cov,atol=1e-10)
assert np.all(pl['var_f']<=ps['var_f']+1e-10)
np.testing.assert_allclose(pl['var_y']-pl['var_f'],.2**2,atol=1e-12)
assert np.linalg.eigvalsh(post['cov_f']).min()>-1e-8
assert np.isfinite(comparison[['rmse','mean_nll','coverage_95']].to_numpy()).all()
assert len(pd.read_csv(exported))==4
print('W7_LAB_B_RUNTIME_CHECKS_PASSED')


### Open the interactive application

In Colab the next cell launches the app automatically. Start with **Fit and compare**, then switch the displayed interval between latent f and future y. Open the likelihood tab and calculate one valid loss. Keep the data settings fixed for a model comparison. Download your CSV and finish the worksheet with an evidence-bounded conclusion. A Gradio share link lasts only while its runtime is active; keep the notebook link.

For a local notebook, set `LAUNCH_WEB_APP=True` to launch. A local or Colab execution marker is not by itself proof that the browser UI was inspected.


In [ ]:
import os
IN_COLAB=bool(os.environ.get('COLAB_RELEASE_TAG')) or 'google.colab' in sys.modules
LAUNCH_WEB_APP=IN_COLAB
demo=create_app(extra=student_extra)
if LAUNCH_WEB_APP:
    demo.queue(default_concurrency_limit=1).launch(share=IN_COLAB,inline=True,prevent_thread_lock=True)
else:
    print('App constructed. Set LAUNCH_WEB_APP=True and rerun to open locally.')
